# Lattice ABP MIPS Sanity Check

This notebook checks whether exact-Gillespie lattice ABP trajectories show Motility-Induced Phase Separation (MIPS).  It records snapshots from a high-persistence regime and tracks coarse density contrast, largest cluster fraction, low-k structure enhancement, and jammed fraction.

In [ ]:
import os
import sys
import time

candidate_roots = [
    os.environ.get("CNEEP_V2_ROOT"),
    os.path.abspath(".."),
    os.path.abspath("."),
    "/home/user1/CNEEP_v2",
]

CNEEP_V2_ROOT = None
for candidate in candidate_roots:
    if candidate and os.path.exists(os.path.join(candidate, "data", "lattice_abp", "core.py")):
        CNEEP_V2_ROOT = candidate
        break

if CNEEP_V2_ROOT is None:
    raise RuntimeError("Could not locate CNEEP_v2 root. Set CNEEP_V2_ROOT.")

if CNEEP_V2_ROOT not in sys.path:
    sys.path.append(CNEEP_V2_ROOT)

print("CNEEP_v2 root:", CNEEP_V2_ROOT)

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from tqdm import tqdm

from data.lattice_abp.core import LatticeABP
from data.lattice_abp.mips_analysis import (
    coarse_density_periodic,
    mips_pass,
    summarize_mips_snapshot,
)

## 1. Configuration

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

abp_params = dict(
    L=64,
    v_plus=10.0,
    v_zero=0.5,
    v_minus=0.1,
    D_rot=0.2,
    density=0.55,
    bc_mode="periodic",
    device=device,
    seed=123,
)

# Snapshot controls.  Increase n_snapshots or steps_per_snapshot if the final
# verdict is borderline on a slow-coarsening seed.
n_snapshots = 400
steps_per_snapshot = 150
coarse_box = 8

output_dir = os.path.join(CNEEP_V2_ROOT, "results", "sanity_lattice_abp_mips")
os.makedirs(output_dir, exist_ok=True)

Pe = (abp_params["v_plus"] - abp_params["v_minus"]) / (2 * abp_params["D_rot"])
print(f"Device: {device}")
print(f"Output: {output_dir}")
print(f"L={abp_params['L']}, density={abp_params['density']}, Pe={Pe:.2f}")
print(f"Total exact Gillespie events per ensemble: {n_snapshots * steps_per_snapshot:,}")

## 2. Run Exact Gillespie Simulation

In [ ]:
sim = LatticeABP(**abp_params)
O, E = sim.init_state(B=1)

frames_occ = []
frames_jammed = []
times = []
summaries = []
jammed_fraction = []

sim_time = 0.0
N = int(abp_params["density"] * abp_params["L"] ** 2)

def record_snapshot():
    jammed = sim.compute_jammed_mask(O, E)
    occ_np = O[0].detach().cpu().numpy().copy()
    jam_np = jammed[0].detach().cpu().numpy().copy()
    n_jammed = int((jammed[0] & (O[0] == 1)).sum().item())

    frames_occ.append(occ_np)
    frames_jammed.append(jam_np)
    times.append(sim_time)
    jammed_fraction.append(n_jammed / max(N, 1))
    summaries.append(
        summarize_mips_snapshot(
            occ_np,
            density=abp_params["density"],
            coarse_box=coarse_box,
            periodic=True,
        )
    )

record_snapshot()

print("Running exact Gillespie events...")
t0 = time.time()
with torch.inference_mode():
    for snap in tqdm(range(1, n_snapshots + 1)):
        for _ in range(steps_per_snapshot):
            O, E, dt_vec = sim.gillespie_step(O, E)
            sim_time += float(dt_vec[0].item())
        record_snapshot()

elapsed = time.time() - t0
print(f"Done in {elapsed:.1f}s")
print(f"Saved snapshots: {len(frames_occ)}")
print(f"Final Gillespie time: {times[-1]:.4f}")

## 3. Scalar MIPS Diagnostics

In [ ]:
metric_names = [
    "largest_cluster_fraction",
    "coarse_std_ratio",
    "coarse_q10",
    "coarse_q90",
    "low_k_ratio",
]

metrics = {name: np.array([s[name] for s in summaries]) for name in metric_names}
metrics["jammed_fraction"] = np.array(jammed_fraction)
metrics["coarse_contrast"] = metrics["coarse_q90"] - metrics["coarse_q10"]
times_arr = np.array(times)

initial = summaries[0]
final = summaries[-1]
passed = mips_pass(initial, final)
cluster_growth = final["largest_cluster_fraction"] / (initial["largest_cluster_fraction"] + 1e-12)

print("Initial summary:")
for key in metric_names:
    print(f"  {key:26s}: {initial[key]:.6f}")
print("\nFinal summary:")
for key in metric_names:
    print(f"  {key:26s}: {final[key]:.6f}")
print(f"  coarse_contrast           : {final['coarse_q90'] - final['coarse_q10']:.6f}")
print(f"  cluster_growth            : {cluster_growth:.3f}x")
print()
print("MIPS sanity verdict:", "PASS" if passed else "BORDERLINE / FAIL")

fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True)
axes = axes.ravel()

axes[0].plot(times_arr, metrics["largest_cluster_fraction"], color="darkorange")
axes[0].set_ylabel("Largest cluster / particles")
axes[0].set_title("Cluster Growth")

axes[1].plot(times_arr, metrics["coarse_std_ratio"], color="steelblue")
axes[1].axhline(1.0, color="gray", ls="--", lw=1)
axes[1].set_ylabel("Observed / random")
axes[1].set_title("Coarse Density Std")

axes[2].plot(times_arr, metrics["coarse_contrast"], color="mediumseagreen")
axes[2].set_ylabel("q90 - q10")
axes[2].set_xlabel("Gillespie time")
axes[2].set_title("Coarse Density Contrast")

axes[3].plot(times_arr, metrics["low_k_ratio"], color="crimson", label="low-k ratio")
axes[3].plot(times_arr, metrics["jammed_fraction"], color="black", alpha=0.6, label="jammed fraction")
axes[3].set_xlabel("Gillespie time")
axes[3].legend()
axes[3].set_title("Structure / Jamming")

for ax in axes:
    ax.grid(alpha=0.25)

plt.tight_layout()
plt.savefig(f"{output_dir}/mips_scalar_diagnostics.png", dpi=150)
plt.show()

## 4. Snapshots and Coarse Density

In [ ]:
indices = [0, len(frames_occ) // 2, len(frames_occ) - 1]
labels = ["initial", "middle", "final"]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for col, (idx, label) in enumerate(zip(indices, labels)):
    occ = frames_occ[idx]
    coarse = coarse_density_periodic(occ, box=coarse_box)

    im0 = axes[0, col].imshow(occ.T, origin="lower", cmap="Greys", vmin=0, vmax=1)
    axes[0, col].set_title(f"{label}: occupancy\nt={times[idx]:.2f}")
    axes[0, col].set_xticks([])
    axes[0, col].set_yticks([])

    im1 = axes[1, col].imshow(coarse.T, origin="lower", cmap="viridis", vmin=0, vmax=1)
    axes[1, col].set_title(f"{label}: coarse density")
    axes[1, col].set_xticks([])
    axes[1, col].set_yticks([])
    fig.colorbar(im1, ax=axes[1, col], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig(f"{output_dir}/mips_snapshots_coarse_density.png", dpi=150)
plt.show()

## 5. Density Histograms

In [ ]:
initial_coarse = coarse_density_periodic(frames_occ[0], box=coarse_box).ravel()
final_coarse = coarse_density_periodic(frames_occ[-1], box=coarse_box).ravel()

fig, ax = plt.subplots(figsize=(8, 5))
bins = np.linspace(0, 1, 41)
ax.hist(initial_coarse, bins=bins, alpha=0.55, density=True, label="initial")
ax.hist(final_coarse, bins=bins, alpha=0.55, density=True, label="final")
ax.axvline(abp_params["density"], color="black", ls="--", lw=1, label="global density")
ax.set_xlabel("Coarse local density")
ax.set_ylabel("Probability density")
ax.set_title("Local Density Histogram")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(f"{output_dir}/mips_density_histogram.png", dpi=150)
plt.show()

print("A clear MIPS run should broaden or split the final histogram relative to the initial one.")

## 6. Final Verdict

In [ ]:
criteria = {
    "largest_cluster_fraction >= 0.35": final["largest_cluster_fraction"] >= 0.35,
    "cluster_growth >= 1.25": cluster_growth >= 1.25,
    "coarse_std_ratio >= 1.35": final["coarse_std_ratio"] >= 1.35,
    "coarse_contrast >= 0.18": (final["coarse_q90"] - final["coarse_q10"]) >= 0.18,
}

for name, ok in criteria.items():
    print(f"{name:34s}: {'PASS' if ok else 'NO'}")

print()
if all(criteria.values()):
    print("PASS: MIPS-like aggregation is visible in this run.")
else:
    print("BORDERLINE / FAIL: increase n_snapshots, steps_per_snapshot, density, or Pe and rerun.")